In [6]:
TEST_COL = ['sex', 'length','diameter','height','whole_weight','shucked_weight','viscera_weight','shell_weight']

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

# 1. Carregar dataset
df = pd.read_csv('abalone_dataset.csv')

# 2. Selecionar atributos (Conjunto otimizado: Pesos + Altura, sem 'sex')
features = ['sex', 'length','diameter','height','whole_weight','shucked_weight','viscera_weight','shell_weight']

if 'sex' in features:
    df = pd.get_dummies(df, columns=['sex'], drop_first=True)

X = df[features]
y = df['type']

# 3. Criar o Pipeline com Padronização e K-NN
# A padronização (StandardScaler) é OBRIGATÓRIA para o K-NN pois ele calcula distâncias
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

# 4. Definir a grade de hiperparâmetros a testar
# K de 1 a 31 (números ímpares para evitar empates)
param_grid = {
    'knn__n_neighbors': range(1, 32, 2),
    'knn__weights': ['uniform', 'distance'],  # 'distance' dá mais peso a vizinhos mais próximos
    'knn__metric': ['euclidean', 'manhattan'] # Métricas de distância
}

# 5. Configurar a Validação Cruzada Estratificada (K=5)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=skf,
    scoring='f1_macro', # Otimiza diretamente pelo F1-Score Macro
    n_jobs=-1
)

# 6. Executar o treinamento e busca do melhor K
grid_search.fit(X, y)

print("=" * 60)
print("  MELHORES HIPERPARÂMETROS PARA O K-NN")
print("=" * 60)
print(f"Melhor K (n_neighbors): {grid_search.best_params_['knn__n_neighbors']}")
print(f"Melhor Peso (weights): {grid_search.best_params_['knn__weights']}")
print(f"Melhor Métrica de Distância: {grid_search.best_params_['knn__metric']}")
print(f"Melhor F1-Score Macro (CV): {grid_search.best_score_ * 100:.2f}%")

# 7. Avaliação detalhada do melhor modelo com cross_validate
best_model = grid_search.best_estimator_
scoring_metrics = ['accuracy', 'f1_macro', 'f1_weighted']

cv_results = cross_validate(best_model, X, y, cv=skf, scoring=scoring_metrics)

print("\n" + "=" * 60)
print("  DESEMPENHO GLOBAL DO K-NN OTIMIZADO")
print("=" * 60)
print(f"Acurácia Média: {cv_results['test_accuracy'].mean() * 100:.2f}% (± {cv_results['test_accuracy'].std() * 100:.2f}%)")
print(f"F1-Score Macro Média: {cv_results['test_f1_macro'].mean() * 100:.2f}% (± {cv_results['test_f1_macro'].std() * 100:.2f}%)")
print(f"F1-Score Ponderado Média: {cv_results['test_f1_weighted'].mean() * 100:.2f}% (± {cv_results['test_f1_weighted'].std() * 100:.2f}%)")

# 8. Gráfico de Evolução do F1-Score em função de K
results_df = pd.DataFrame(grid_search.cv_results_)
best_metric_df = results_df[
    (results_df['param_knn__metric'] == grid_search.best_params_['knn__metric']) &
    (results_df['param_knn__weights'] == grid_search.best_params_['knn__weights'])
]

plt.figure(figsize=(10, 5))
plt.plot(best_metric_df['param_knn__n_neighbors'], best_metric_df['mean_test_score'] * 100, marker='o', color='purple')
plt.title(f"Ajuste do Valor de K (Métrica: {grid_search.best_params_['knn__metric']}, Pesos: {grid_search.best_params_['knn__weights']})")
plt.xlabel("Número de Vizinhos (K)")
plt.ylabel("F1-Score Macro CV (%)")
plt.grid(True)
plt.show()

ValueError: 
All the 320 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
320 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Computaria\archives\ML\.venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 855, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Computaria\archives\ML\.venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Computaria\archives\ML\.venv\Lib\site-packages\sklearn\pipeline.py", line 632, in fit
    Xt = self._fit(
        X, y, routed_params, raw_params=params, callback_ctx=callback_ctx
    )
  File "c:\Computaria\archives\ML\.venv\Lib\site-packages\sklearn\pipelin